In [1]:
import pandas as pd
import plotly.graph_objects as go

# Load data from Excel
excel_file = 'OVERALL RESULT.xlsx'
sheet_name = 'JA&MB'
df = pd.read_excel(excel_file, sheet_name=sheet_name)
# Define the order of methods
# method_order = ['PR-DDQN', 'KeyRef2', 'CDR1', 'CDR2', 'CDR3', 'CDR4', 'CDR5', 'CDR6']
method_order = ['PR-DDQN', "Modified Luo's DDQN"]

df['Method'] = pd.Categorical(df['Method'], categories=method_order)
# Create a boxplot
fig = go.Figure()

# Add boxplot trace
fig.add_trace(go.Box(
    x=df['Method'], 
    y=df['Tardiness'],
    boxpoints='outliers',
    fillcolor='white',
    marker=dict(color='blue'),
    line=dict(color='blue'),
    name='Tardiness'
))

# Calculate mean for each Method and add as a scatter trace
mean_values = df.groupby('Method')['Tardiness'].mean().reset_index()

fig.add_trace(go.Scatter(
    x=mean_values['Method'],
    y=mean_values['Tardiness'],
    mode='lines+markers',
    line=dict(color='red'),
    marker=dict(color='red', size=8),
    name='Mean Tardiness'
))

# Customize layout
fig.update_layout(
    title='Tardiness Distribution by Method',
    xaxis_title='Method',
    yaxis_title='Tardiness (seconds)',
    xaxis=dict(
        tickfont=dict(size=14),  # Adjust the font size of x-axis labels here
        categoryorder='array',   # Ensure categorical order is respected
        categoryarray=method_order
    ),
    template='plotly_white'
)

# Show the plot
fig.show()


C:\Users\ASUS\AppData\Local\Temp\ipykernel_500\558436305.py:28: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [2]:
import pandas as pd
import plotly.graph_objects as go

# Calculate median tardiness for each method
median_values = df.groupby('Method')['Tardiness'].median().reset_index()

# Calculate the minimum tardiness within each group defined by InstanceID and ScenarioID
grouped = df.groupby(['InstanceID', 'ScenarioID'])

# Initialize a counter for each method
method_min_counts = pd.Series(0, index=df['Method'].unique())

# Count the minimum occurrences per method in each group
for name, group in grouped:
    min_tardiness = group['Tardiness'].min()
    min_methods = group[group['Tardiness'] == min_tardiness]['Method'].unique()
    for method in min_methods:
        method_min_counts[method] += 1

# Convert the counts to a DataFrame
min_counts = method_min_counts.reset_index()
min_counts.columns = ['Method', 'Count']

# Total number of InstanceID and ScenarioID combinations
total_combinations = len(grouped)

# Calculate the percentage
min_counts['Percentage'] = (min_counts['Count'] / total_combinations) * 100

# Reorder median values and min percentages according to the specified order
median_values['Method'] = pd.Categorical(median_values['Method'], categories=method_order, ordered=True)
median_values = median_values.sort_values('Method')

min_counts['Method'] = pd.Categorical(min_counts['Method'], categories=method_order, ordered=True)
min_counts = min_counts.sort_values('Method')

# Create the combined plot
fig = go.Figure()

# Add line plot for median values (add this first to ensure it's on top)
fig.add_trace(go.Scatter(
    x=median_values['Method'],
    y=median_values['Tardiness'],
    mode='lines+markers',
    line=dict(color='red'),
    marker=dict(color='red', size=8),
    name='Median Tardiness',
    yaxis='y2'
))

# Add bar plot for percentage of minimum values
fig.add_trace(go.Bar(
    x=min_counts['Method'],
    y=min_counts['Percentage'],
    name='Win rate',
    marker=dict(color='white', line=dict(color='blue', width=2)),
    width=0.3  # Adjust the width of the columns to make them thinner
))

# Customize layout
fig.update_layout(
    title='Win rate and Median of tardiness by Method',
    xaxis_title='Method',
    yaxis=dict(
        title='Percentage (%)',
        titlefont=dict(color='black'),  # Change title font color to black
        tickfont=dict(color='black'),  # Change tick font color to black
        range=[0, 100]  # Ensure the y-axis starts at 0
    ),
    yaxis2=dict(
        title='Tardiness (seconds)',
        titlefont=dict(color='black'),  # Change title font color to black
        tickfont=dict(color='black'),  # Change tick font color to black
        overlaying='y',
        side='right',
        range=[0, median_values['Tardiness'].max() + 100],  # Ensure the y-axis starts at 0 and has a reasonable max value
        showgrid=False  # Hide the horizontal grid lines
    ),
    template='plotly_white',
    xaxis=dict(
        tickfont=dict(size=14)  # Adjust the font size of x-axis labels here
    ),
    font=dict(color='black'),  # Change the font color of the entire plot to black
    legend=dict(
        x=1.1,
        y=1,
        bgcolor='rgba(255, 255, 255, 0)',
        bordercolor='rgba(0, 0, 0, 0)'
    )
)

# Show the plot
fig.show()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_500\3190877810.py:5: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

